# Part 3 – Merge by ubigeo and analysis
**Group 2 · Season 2023 (January 1 to May 31)**

Research question: Does the State declare emergencies where it rains the most?

In [3]:
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## 1. Create the ubigeo table

We create a table with the official department-level ubigeo codes from INEI (25 departments, including Callao). Each ubigeo is written between quotes so it is stored as **text** and keeps the leading zero ("01", not 1).

Source: official department-level ubigeo codes from INEI, as provided in the assignment instructions.

In [4]:
# Official INEI department codes (from the assignment instructions)
ubigeos = pd.DataFrame({
    "ubigeo": ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10",
               "11", "12", "13", "14", "15", "16", "17", "18", "19", "20",
               "21", "22", "23", "24", "25"],
    "departamento": ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
                     "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
                     "Ica", "Junín", "La Libertad", "Lambayeque", "Lima",
                     "Loreto", "Madre de Dios", "Moquegua", "Pasco", "Piura",
                     "Puno", "San Martín", "Tacna", "Tumbes", "Ucayali"]
})

ubigeos.to_csv("datos/ubigeos.csv", index=False, encoding="utf-8-sig")

print(ubigeos.shape)
ubigeos.head()

(25, 2)


,ubigeo,departamento
0,01,Amazonas
1,02,Áncash
2,03,Apurímac
3,04,Arequipa
4,05,Ayacucho


## 2. Reading the ubigeo file in two ways

We read `ubigeos.csv` twice: first with a normal `pd.read_csv`, and then telling pandas that the `ubigeo` column is text (`dtype={"ubigeo": str}`).

In [9]:
# Normal reading
ubigeos_normal = pd.read_csv("datos/ubigeos.csv", encoding="utf-8-sig")

# Reading ubigeo as text
ubigeos_texto = pd.read_csv("datos/ubigeos.csv", encoding="utf-8-sig", dtype={"ubigeo": str})

# Type of the ubigeo column in each case
print("Normal reading:", ubigeos_normal["ubigeo"].dtype)
print("Reading as text:", ubigeos_texto["ubigeo"].dtype)

# Side by side comparison
comparacion = pd.DataFrame({
    "departamento": ubigeos_texto["departamento"],
    "ubigeo_normal": ubigeos_normal["ubigeo"],
    "ubigeo_texto": ubigeos_texto["ubigeo"]
})
comparacion.head(10)

Normal reading: int64
Reading as text: str


,departamento,ubigeo_normal,ubigeo_texto
0,Amazonas,1,01
1,Áncash,2,02
2,Apurímac,3,03
3,Arequipa,4,04
4,Ayacucho,5,05
5,Cajamarca,6,06
6,Callao,7,07
7,Cusco,8,08
8,Huancavelica,9,09
9,Huánuco,10,10


In [10]:
# Test: does "01" match 1?
print(ubigeos_texto["ubigeo"][0] == ubigeos_normal["ubigeo"][0])

# Test: even converting the number to text, "1" is not "01"
print(ubigeos_texto["ubigeo"][0] == str(ubigeos_normal["ubigeo"][0]))

False
False


### Why does it matter?

With a normal `pd.read_csv`, pandas guesses that `ubigeo` is a number (`int64`) and **drops the leading zero**: Amazonas goes from `01` to `1`. When we use `dtype={"ubigeo": str}`, the column is read as text and keeps its original 2 digits.

This matters for two reasons:

1. **The ubigeo is a code, not a quantity.** It works like an ID (like a DNI number): it makes no sense to add or average ubigeos. It must keep its official format (2 digits for departments, 6 for districts, e.g. Chachapoyas is `010101`, not `10101`).
2. **The merge would fail.** In step 5 we merge the tables by `ubigeo`. As the test above shows, `"01"` and `1` are not equal for Python, and even converting the number back to text gives `"1"`, not `"01"`. If one table has the code as text and the other as a number, departments would not match and would be lost or left empty.

That is why we always read the ubigeo with `dtype={"ubigeo": str}`.

## 3. Adding the ubigeo to the decrees and rainfall tables

We load the two tables created in Parts 1 and 2 and check how the department names are written, since we will use them to add the ubigeo.

In [11]:
decretos = pd.read_csv("datos/decretos_por_departamento.csv", encoding="utf-8-sig")
lluvias = pd.read_csv("datos/lluvias_por_departamento.csv", encoding="utf-8-sig")

print("Decrees:", decretos.shape)
print("Rainfall:", lluvias.shape)

print("\nDepartment names in decrees:")
print(decretos["departamento"].tolist())

print("\nDepartment names in rainfall:")
print(lluvias["departamento"].tolist())

Decrees: (25, 3)
Rainfall: (25, 6)

Department names in decrees:
['Amazonas', 'Ancash', 'Apurimac', 'Arequipa', 'Ayacucho', 'Cajamarca', 'Callao', 'Cusco', 'Huancavelica', 'Huanuco', 'Ica', 'Junin', 'La Libertad', 'Lambayeque', 'Lima', 'Loreto', 'Madre de Dios', 'Moquegua', 'Pasco', 'Piura', 'Puno', 'San Martin', 'Tacna', 'Tumbes', 'Ucayali']

Department names in rainfall:
['Amazonas', 'Ancash', 'Apurimac', 'Arequipa', 'Ayacucho', 'Cajamarca', 'Cusco', 'Huancavelica', 'Huanuco', 'Ica', 'Junin', 'La Libertad', 'Lambayeque', 'Lima', 'Loreto', 'Madre de Dios', 'Moquegua', 'Pasco', 'Piura', 'Puno', 'San Martin', 'Tacna', 'Tumbes', 'Ucayali', 'Callao']


In [12]:
# First try: add the ubigeo by matching the department name as it is
decretos_ubigeo = decretos.merge(ubigeos_texto, on="departamento", how="left")
lluvias_ubigeo = lluvias.merge(ubigeos_texto, on="departamento", how="left")

# Departments that did not find their ubigeo
print("Decrees without ubigeo:")
print(decretos_ubigeo[decretos_ubigeo["ubigeo"].isna()]["departamento"].tolist())

print("\nRainfall without ubigeo:")
print(lluvias_ubigeo[lluvias_ubigeo["ubigeo"].isna()]["departamento"].tolist())

Decrees without ubigeo:
['Ancash', 'Apurimac', 'Huanuco', 'Junin', 'San Martin']

Rainfall without ubigeo:
['Ancash', 'Apurimac', 'Huanuco', 'Junin', 'San Martin']


## 4. Fixing department names without ubigeo

The first merge shows that 5 departments did not find their ubigeo in either table: Ancash, Apurimac, Huanuco, Junin and San Martin. The reason is that Parts 1 and 2 wrote the names **without accents**, while the official INEI table uses accents (Áncash, Apurímac, Huánuco, Junín, San Martín). For Python, "Ancash" and "Áncash" are different texts.

We fix them with a dictionary that replaces each name with its official INEI spelling, and then we check again.

In [13]:
# Dictionary: name as written in Parts 1 and 2 -> official INEI name
correcciones = {
    "Ancash": "Áncash",
    "Apurimac": "Apurímac",
    "Huanuco": "Huánuco",
    "Junin": "Junín",
    "San Martin": "San Martín"
}

decretos["departamento"] = decretos["departamento"].replace(correcciones)
lluvias["departamento"] = lluvias["departamento"].replace(correcciones)

# Second try: add the ubigeo again
decretos_ubigeo = decretos.merge(ubigeos_texto, on="departamento", how="left")
lluvias_ubigeo = lluvias.merge(ubigeos_texto, on="departamento", how="left")

print("Decrees without ubigeo:")
print(decretos_ubigeo[decretos_ubigeo["ubigeo"].isna()]["departamento"].tolist())

print("\nRainfall without ubigeo:")
print(lluvias_ubigeo[lluvias_ubigeo["ubigeo"].isna()]["departamento"].tolist())

Decrees without ubigeo:
[]

Rainfall without ubigeo:
[]
